## Quick start

If you have already worked through the first parts and are just repeating the "lifecycle" parts from scratch, you can use this notebook to bootstrap the first parts (provision new resources, install Kubernetes, and set up Argo CD)! Note: you still need to have installed Terraform + Ansible + Kubespray + set up your `clouds.yaml`, `ansible.cfg` etc. or it won't work.

This quick start is for "expert" users - it's not guaranteed to be robust to things like timing, etc.

In [ ]:
# runs in Chameleon Jupyter environment
export PATH=/work/.local/bin:$PATH
export PYTHONUSERBASE=/work/.local

export NET_ID=netID
export OS_AUTH_URL=https://kvm.tacc.chameleoncloud.org:5000/v3
export OS_PROJECT_NAME="CHI-XXXXXX"
export OS_REGION_NAME="KVM@TACC"

In [ ]:
# runs in Chameleon Jupyter environment
openstack reservation lease create lease_mlops_$NET_ID \
  --start-date "$(date -u '+%Y-%m-%d %H:%M')" \
  --end-date "$(date -u -d '+12 hours' '+%Y-%m-%d %H:%M')" \
  --reservation "resource_type=flavor:instance,flavor_id=$(openstack flavor show m1.large -f value -c id),amount=3"

In [ ]:
# runs in Chameleon Jupyter environment
export TF_VAR_suffix=$NET_ID
export TF_VAR_key=id_rsa_chameleon
export TF_VAR_reservation=$(openstack reservation lease show lease_mlops_$NET_ID -f json -c reservations \
      | jq -r '.reservations[0].flavor_id')

In [ ]:
# runs in Chameleon Jupyter environment
unset $(set | grep -o "^OS_[A-Za-z0-9_]*")
cd /work/gourmetgram-iac/tf/kvm
terraform apply -auto-approve
# update ansible.cfg with correct floating IP here! then copy it to /work/gourmetgram-iac/ansible/ansible.cfg

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible-playbook -i inventory.yml pre_k8s/pre_k8s_configure.yml

In [ ]:
# runs in Chameleon Jupyter environment
export ANSIBLE_CONFIG=/work/gourmetgram-iac/ansible/ansible.cfg
export ANSIBLE_ROLES_PATH=roles
cd /work/gourmetgram-iac/ansible/k8s/kubespray
ansible-playbook -i ../inventory/mycluster --become --become-user=root ./cluster.yml

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible-playbook -i inventory.yml post_k8s/post_k8s_configure.yml

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible-playbook -i inventory.yml argocd/argocd_add_platform.yml

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/ansible
ansible-playbook -i inventory.yml argocd/workflow_build_init.yml
ansible-playbook -i inventory.yml argocd/argocd_add_staging.yml
ansible-playbook -i inventory.yml argocd/argocd_add_canary.yml
ansible-playbook -i inventory.yml argocd/argocd_add_prod.yml
ansible-playbook -i inventory.yml argocd/workflow_templates_apply.yml

In [ ]:
# runs in Chameleon Jupyter environment
cd /work/gourmetgram-iac/tf/kvm
# un-comment to bring down resources
# terraform destroy -auto-approve